<a href="https://colab.research.google.com/github/mnpmangata/bookings/blob/main/pvcalculator_ph_032626.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# %% [markdown]
# # Solar PV System Sizing Tool for Philippine and Chinese Markets
# ### For Commercial and Residential Applications
#
# This notebook helps you size solar PV systems based on your energy consumption or power load requirements.
#
# **New sizing approach:**
# 1. Size inverter based on load/consumption
# 2. Size battery based on load and autonomy
# 3. Size solar panels so that DC power = inverter AC × AC/DC ratio

# %%
# Import necessary libraries
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime
import io

# Enable matplotlib inline
%matplotlib inline

# Detect if running in Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# %% [markdown]
# ## Equipment Database
# ### Based on available brands in Philippines and China
#
# **Important:** The following three Excel files must be provided:
# - `solar_panels.xlsx`
# - `inverters.xlsx`
# - `batteries.xlsx`
#
# If you are running in **Google Colab**, you will be prompted to upload them when the cell below is executed.

# %%
# ----------------------------------------------------------------------
# Load equipment databases strictly from Excel files (with upload in Colab)
# ----------------------------------------------------------------------

def load_equipment(file_name, required_columns):
    """Load equipment data from an Excel file and validate required columns."""
    # In Colab, check if the file already exists in the current directory
    if not os.path.exists(file_name):
        if IN_COLAB:
            print(f"📂 File '{file_name}' not found. Please upload it now.")
            uploaded = files.upload()
            if file_name not in uploaded:
                raise FileNotFoundError(f"❌ Upload failed. '{file_name}' not uploaded.")
        else:
            raise FileNotFoundError(f"❌ Required file '{file_name}' not found in the current directory.")

    try:
        df = pd.read_excel(file_name)
    except Exception as e:
        raise Exception(f"❌ Error reading '{file_name}': {e}")

    missing_cols = [col for col in required_columns if col not in df.columns]
    if missing_cols:
        raise ValueError(f"❌ File '{file_name}' is missing required columns: {missing_cols}")

    # Convert 'Type' column to string to avoid sorting issues later
    if 'Type' in df.columns:
        df['Type'] = df['Type'].astype(str).fillna('Unknown')

    return df

# Define expected columns for each equipment type
solar_columns = ['Brand', 'Type', 'Country', 'Model', 'Power_Range_W',
                 'Efficiency', 'Vmp_V', 'Imp_A', 'Voc_V', 'Isc_A', 'Price_PHP_W']
inverter_columns = ['Brand', 'Country', 'Type', 'Power_kW', 'Efficiency',
                    'MPPT_Inputs', 'Max_DC_Voltage', 'MPPT_Voltage_Range', 'Price_PHP_kW']
battery_columns = ['Brand', 'Country', 'Type', 'Capacity_kWh', 'Voltage_V',
                   'DoD', 'Cycle_Life', 'Price_PHP_kWh']

# Load the data
solar_panels = load_equipment('solar_panels.xlsx', solar_columns)
inverters    = load_equipment('inverters.xlsx',    inverter_columns)
batteries    = load_equipment('batteries.xlsx',    battery_columns)

print("✅ Equipment data loaded successfully from Excel files.")

# %%
# System Parameters
system_voltages = [12, 24, 48, 120, 240, 380, 480]
system_types = ['On-Grid', 'Off-Grid', 'Hybrid']
customer_types = ['Residential', 'Commercial', 'Industrial']

# %% [markdown]
# ## BOQ Calculator Functions

# %%
def calculate_mounting_structure(num_panels, panel_area, roof_type='tile'):
    """
    Calculate mounting structure requirements
    Returns dictionary with quantities and costs
    """
    # Standard mounting components per panel (based on typical Philippine installations)
    mounting = {
        'End_Clamps': num_panels * 2,  # 2 end clamps per row
        'Mid_Clamps': num_panels * 2,  # 2 mid clamps per panel (shared between panels)
        'Rail_Meters': num_panels * 3.5,  # 3.5m of rail per panel average
        'Rail_Splices': max(1, int(num_panels / 10)),  # 1 splice per 10 panels
        'L_Feet': num_panels * 4,  # 4 L-feet per panel
        'Flashings': num_panels * 4,  # 4 flashings per panel
        'T_Bolts': num_panels * 8,  # 8 T-bolts per panel
        'Spring_Nuts': num_panels * 8,  # 8 spring nuts per panel
        'Grounding_Lugs': num_panels,  # 1 grounding lug per panel
        'Wire_Clips': num_panels * 10,  # 10 wire clips per panel
    }

    # Adjust for roof type
    if roof_type == 'metal':
        mounting['L_Feet'] = num_panels * 2  # Metal roof uses fewer feet
        mounting['Flashings'] = num_panels * 2
    elif roof_type == 'concrete':
        # Add concrete anchors for concrete roof
        mounting['Concrete_Anchors'] = num_panels * 4  # NOTE: num_palets? Probably meant num_panels
        mounting['Concrete_Anchors'] = num_panels * 4

    return mounting

def calculate_dc_wiring(strings_per_mppt, num_panels, panel_current, string_voltage, distance=20):
    """
    Calculate DC wiring requirements
    """
    # PV wire (from panels to combiner box)
    pv_wire_length = strings_per_mppt * distance * 2  # Positive and negative

    # String combiner boxes
    num_combiners = max(1, int(np.ceil(strings_per_mppt / 4)))  # 4 strings per combiner typically

    # MC4 connectors
    mc4_pairs = num_panels * 2  # 2 pairs per panel (input/output)

    # DC isolators
    dc_isolators = num_combiners + 1  # One per combiner + main DC isolator

    return {
        'PV_Wire_m': pv_wire_length,
        'String_Combiners': num_combiners,
        'MC4_Pairs': mc4_pairs,
        'DC_Isolators': dc_isolators,
        'DC_SPD': num_combiners,  # Surge protection devices
        'DC_Fuses': strings_per_mppt * 2,  # Fuses for each string positive and negative
        'DC_Breakers': num_combiners,  # Main DC breakers
    }

def calculate_ac_wiring(inverter_power_kw, distance_to_grid=15):
    """
    Calculate AC wiring requirements based on inverter size
    """
    # Determine cable size based on power
    if inverter_power_kw <= 5:
        cable_size = '6mm²'
        cable_price_per_m = 120
    elif inverter_power_kw <= 10:
        cable_size = '10mm²'
        cable_price_per_m = 180
    elif inverter_power_kw <= 20:
        cable_size = '16mm²'
        cable_price_per_m = 250
    elif inverter_power_kw <= 50:
        cable_size = '35mm²'
        cable_price_per_m = 450
    else:
        cable_size = '70mm²'
        cable_price_per_m = 800

    ac_cable_length = distance_to_grid * 3  # 3-phase or 3 wires

    return {
        'AC_Cable_Size': cable_size,
        'AC_Cable_m': ac_cable_length,
        'AC_Breaker': 1,  # Main AC breaker
        'AC_SPD': 1,  # AC surge protection
        'AC_Isolator': 1,  # AC isolator switch
        'Metering': 1,  # Export/import meter for grid-tied
        'Cable_Lugs': 6,  # 6 lugs for terminations
    }

def calculate_protection_devices(system_type, battery_info=None):
    """
    Calculate protection devices based on system type
    """
    protection = {
        'DC_Circuit_Breakers': 2,  # Main DC breakers
        'AC_Circuit_Breakers': 2,  # Main AC breakers
        'Surge_Protection_DC': 2,  # DC SPDs
        'Surge_Protection_AC': 2,  # AC SPDs
        'Grounding_Rods': 2,  # Grounding electrodes
        'Grounding_Cable_m': 15,  # Grounding wire
        'Grounding_Clamps': 4,  # Grounding clamps
        'Warning_Signs': 4,  # Safety warning signs
        'Fire_Extinguisher': 1,  # CO2 extinguisher for electrical
    }

    if system_type in ['Off-Grid', 'Hybrid'] and battery_info:
        protection.update({
            'Battery_Breakers': battery_info['num_batteries'],  # Breaker per battery
            'Battery_Fuses': battery_info['num_batteries'] * 2,  # Fuses per battery
            'Battery_Isolator': 1,  # Main battery isolator
            'Battery_Temperature_Sensor': 1,
        })

    return protection

def calculate_monitoring_system(system_type):
    """
    Calculate monitoring system components
    """
    monitoring = {
        'Energy_Meter': 1,  # Energy monitoring device
        'CT_Sensors': 3,  # Current transformers for 3-phase
        'Data_Logger': 1,  # Data logging device
        'Remote_Monitoring_Kit': 1,  # WiFi/4G module
        'Display_Unit': 1,  # Local display
        'Temperature_Sensor': 2,  # Ambient and panel temp
        'Irradiance_Sensor': 1,  # Solar irradiance sensor
    }

    return monitoring

def calculate_boq(num_panels, panel_power, inverter_power_kw, battery_info, system_type, daily_cons, selected_panel, inverter_info, roof_type='tile', grid_distance=15):
    """
    Complete Bill of Quantities calculation
    Returns detailed BOQ as DataFrame
    """
    boq_items = []

    # 1. SOLAR PANELS
    boq_items.append({
        'Category': 'Solar Panels',
        'Item': f'Solar Panel - {selected_panel["Brand"]} {panel_power}W',
        'Quantity': num_panels,
        'Unit': 'pcs',
        'Unit_Price': selected_panel['Price_PHP_W'] * panel_power,
        'Total_Price': num_panels * selected_panel['Price_PHP_W'] * panel_power
    })

    # 2. INVERTER
    boq_items.append({
        'Category': 'Inverter',
        'Item': f'{inverter_info["Brand"]} {inverter_power_kw}kW {inverter_info["Type"]} Inverter',
        'Quantity': 1,
        'Unit': 'pc',
        'Unit_Price': inverter_power_kw * inverters[inverters['Brand'] == inverter_info['Brand']].iloc[0]['Price_PHP_kW'],
        'Total_Price': inverter_power_kw * inverters[inverters['Brand'] == inverter_info['Brand']].iloc[0]['Price_PHP_kW']
    })

    # 3. BATTERIES
    if battery_info:
        boq_items.append({
            'Category': 'Batteries',
            'Item': f'{battery_info["battery"]["Brand"]} {battery_info["battery"]["Capacity_kWh"]}kWh Battery',
            'Quantity': battery_info['num_batteries'],
            'Unit': 'pcs',
            'Unit_Price': battery_info['battery']['Price_PHP_kWh'] * battery_info['battery']['Capacity_kWh'],
            'Total_Price': battery_info['total_capacity_kwh'] * battery_info['battery']['Price_PHP_kWh']
        })

    # 4. MOUNTING STRUCTURE
    mounting = calculate_mounting_structure(num_panels, num_panels * 1.7, roof_type)
    mounting_items = [
        ('End Clamps', mounting['End_Clamps'], 45),
        ('Mid Clamps', mounting['Mid_Clamps'], 35),
        ('Mounting Rails (m)', mounting['Rail_Meters'], 150),
        ('Rail Splices', mounting['Rail_Splices'], 80),
        ('L-Feet', mounting['L_Feet'], 60),
        ('Flashings', mounting['Flashings'], 55),
        ('T-Bolts', mounting['T_Bolts'], 12),
        ('Spring Nuts', mounting['Spring_Nuts'], 10),
        ('Grounding Lugs', mounting['Grounding_Lugs'], 25),
        ('Wire Clips', mounting['Wire_Clips'], 5),
    ]

    for item_name, qty, unit_price in mounting_items:
        boq_items.append({
            'Category': 'Mounting Structure',
            'Item': item_name,
            'Quantity': qty,
            'Unit': 'pcs' if 'm' not in item_name else 'meter',
            'Unit_Price': unit_price,
            'Total_Price': qty * unit_price
        })

    # 5. DC CABLES & ACCESSORIES
    # Estimate number of strings
    num_strings = max(1, int(np.ceil(num_panels / 15)))  # 15 panels per string typical
    string_current = selected_panel['Imp_A']
    string_voltage = selected_panel['Vmp_V'] * (num_panels / num_strings)

    dc_wiring = calculate_dc_wiring(num_strings, num_panels, string_current, string_voltage)

    dc_items = [
        ('PV Cable (4mm²) - meter', dc_wiring['PV_Wire_m'], 45),
        ('String Combiners', dc_wiring['String_Combiners'], 3500),
        ('MC4 Connectors (pair)', dc_wiring['MC4_Pairs'], 85),
        ('DC Isolators', dc_wiring['DC_Isolators'], 1200),
        ('DC Surge Protection', dc_wiring['DC_SPD'], 1800),
        ('DC Fuses', dc_wiring['DC_Fuses'], 150),
        ('DC Circuit Breakers', dc_wiring['DC_Breakers'], 950),
    ]

    for item_name, qty, unit_price in dc_items:
        boq_items.append({
            'Category': 'DC Electrical',
            'Item': item_name,
            'Quantity': qty,
            'Unit': 'pcs' if 'meter' not in item_name else 'meter',
            'Unit_Price': unit_price,
            'Total_Price': qty * unit_price
        })

    # 6. AC CABLES & ACCESSORIES
    ac_wiring = calculate_ac_wiring(inverter_power_kw, grid_distance)

    ac_items = [
        (f'AC Cable ({ac_wiring["AC_Cable_Size"]}) - meter', ac_wiring['AC_Cable_m'], 180 if '6mm²' in ac_wiring["AC_Cable_Size"] else 250),
        ('Main AC Breaker', ac_wiring['AC_Breaker'], 2500),
        ('AC Surge Protection', ac_wiring['AC_SPD'], 2200),
        ('AC Isolator', ac_wiring['AC_Isolator'], 1800),
        ('Bidirectional Meter', ac_wiring['Metering'], 8500),
        ('Cable Lugs', ac_wiring['Cable_Lugs'], 85),
    ]

    for item_name, qty, unit_price in ac_items:
        boq_items.append({
            'Category': 'AC Electrical',
            'Item': item_name,
            'Quantity': qty,
            'Unit': 'pcs' if 'meter' not in item_name else 'meter',
            'Unit_Price': unit_price,
            'Total_Price': qty * unit_price
        })

    # 7. PROTECTION DEVICES
    protection = calculate_protection_devices(system_type, battery_info)

    protection_items = [
        ('DC Circuit Breakers', protection['DC_Circuit_Breakers'], 950),
        ('AC Circuit Breakers', protection['AC_Circuit_Breakers'], 1200),
        ('DC SPD', protection['Surge_Protection_DC'], 1800),
        ('AC SPD', protection['Surge_Protection_AC'], 2200),
        ('Grounding Rods', protection['Grounding_Rods'], 450),
        ('Grounding Cable (m)', protection['Grounding_Cable_m'], 65),
        ('Grounding Clamps', protection['Grounding_Clamps'], 85),
        ('Warning Signs', protection['Warning_Signs'], 150),
        ('Fire Extinguisher (CO2)', protection['Fire_Extinguisher'], 3500),
    ]

    if 'Battery_Breakers' in protection:
        protection_items.extend([
            ('Battery Breakers', protection['Battery_Breakers'], 850),
            ('Battery Fuses', protection['Battery_Fuses'], 120),
            ('Battery Isolator', protection['Battery_Isolator'], 2200),
            ('Battery Temperature Sensor', protection['Battery_Temperature_Sensor'], 1800),
        ])

    for item_name, qty, unit_price in protection_items:
        boq_items.append({
            'Category': 'Protection & Safety',
            'Item': item_name,
            'Quantity': qty,
            'Unit': 'pcs' if 'meter' not in item_name else 'meter',
            'Unit_Price': unit_price,
            'Total_Price': qty * unit_price
        })

    # 8. MONITORING SYSTEM
    monitoring = calculate_monitoring_system(system_type)

    monitoring_items = [
        ('Energy Meter', monitoring['Energy_Meter'], 5500),
        ('CT Sensors', monitoring['CT_Sensors'], 1200),
        ('Data Logger', monitoring['Data_Logger'], 6500),
        ('Remote Monitoring Kit', monitoring['Remote_Monitoring_Kit'], 4500),
        ('Display Unit', monitoring['Display_Unit'], 3800),
        ('Temperature Sensor', monitoring['Temperature_Sensor'], 1500),
        ('Irradiance Sensor', monitoring['Irradiance_Sensor'], 8500),
    ]

    for item_name, qty, unit_price in monitoring_items:
        boq_items.append({
            'Category': 'Monitoring System',
            'Item': item_name,
            'Quantity': qty,
            'Unit': 'pcs',
            'Unit_Price': unit_price,
            'Total_Price': qty * unit_price
        })

    # 9. CONSUMABLES & INSTALLATION MATERIALS
    consumables = [
        ('Cable Ties (100pcs pack)', 2, 250),
        ('Cable Markers (set)', 1, 350),
        ('Electrical Tape (roll)', 5, 85),
        ('Conduit Pipe (m)', num_panels * 2, 45),
        ('Conduit Fittings', num_panels * 4, 35),
        ('Junction Boxes', 4, 250),
        ('Weatherproof Sealant (tube)', 3, 280),
        ('Anti-theft bolts', num_panels * 2, 25),
    ]

    for item_name, qty, unit_price in consumables:
        boq_items.append({
            'Category': 'Consumables',
            'Item': item_name,
            'Quantity': qty,
            'Unit': 'pcs' if 'pack' not in item_name else 'pack',
            'Unit_Price': unit_price,
            'Total_Price': qty * unit_price
        })

    # 10. LABOR (calculated separately)
    labor_hours = num_panels * 2.5  # 2.5 hours per panel average
    boq_items.append({
        'Category': 'Labor',
        'Item': 'Installation Labor',
        'Quantity': labor_hours,
        'Unit': 'hours',
        'Unit_Price': 350,  # PHP 350/hour typical rate
        'Total_Price': labor_hours * 350
    })

    boq_items.append({
        'Category': 'Labor',
        'Item': 'Electrical Connection & Commissioning',
        'Quantity': 1,
        'Unit': 'lump sum',
        'Unit_Price': 15000,
        'Total_Price': 15000
    })

    # 11. TRANSPORT & LOGISTICS
    boq_items.append({
        'Category': 'Logistics',
        'Item': 'Delivery & Transport',
        'Quantity': 1,
        'Unit': 'lump sum',
        'Unit_Price': 5000 + (num_panels * 200),  # Base + per panel
        'Total_Price': 5000 + (num_panels * 200)
    })

    boq_items.append({
        'Category': 'Logistics',
        'Item': 'Crane/Equipment Rental (if needed)',
        'Quantity': 1,
        'Unit': 'lump sum',
        'Unit_Price': 8000 if num_panels > 20 else 0,
        'Total_Price': 8000 if num_panels > 20 else 0
    })

    # 12. PERMITS & DOCUMENTATION
    permit_cost = daily_cons * 100 if daily_cons else 0  # Rough estimate based on consumption
    boq_items.append({
        'Category': 'Permits',
        'Item': 'Building Permit & Electrical Inspection',
        'Quantity': 1,
        'Unit': 'lump sum',
        'Unit_Price': permit_cost,
        'Total_Price': permit_cost
    })

    boq_items.append({
        'Category': 'Permits',
        'Item': 'Utility Interconnection Application',
        'Quantity': 1,
        'Unit': 'lump sum',
        'Unit_Price': 5000 if system_type == 'On-Grid' else 0,
        'Total_Price': 5000 if system_type == 'On-Grid' else 0
    })

    # Create DataFrame
    boq_df = pd.DataFrame(boq_items)

    return boq_df

def display_boq_summary(boq_df):
    """
    Display BOQ summary with totals by category
    """
    print("\n" + "="*80)
    print("📋 DETAILED BILL OF QUANTITIES (BOQ)")
    print("="*80)

    # Group by category and calculate totals
    category_totals = boq_df.groupby('Category')['Total_Price'].sum().sort_values(ascending=False)

    # Display summary table
    summary_data = []
    for category, total in category_totals.items():
        summary_data.append({
            'Category': category,
            'Total (₱)': f"₱{total:,.2f}",
            'Percentage': f"{(total/category_totals.sum()*100):.1f}%"
        })

    summary_df = pd.DataFrame(summary_data)
    print("\n📊 COST SUMMARY BY CATEGORY:")
    print(summary_df.to_string(index=False))

    print(f"\n💰 GRAND TOTAL: ₱{category_totals.sum():,.2f}")

    # Display detailed BOQ
    print("\n" + "="*80)
    print("📋 DETAILED ITEMS:")
    print("="*80)

    # Format the detailed BOQ for display
    display_df = boq_df.copy()
    display_df['Unit_Price'] = display_df['Unit_Price'].apply(lambda x: f"₱{x:,.0f}")
    display_df['Total_Price'] = display_df['Total_Price'].apply(lambda x: f"₱{x:,.0f}")

    # Sort by category
    display_df = display_df.sort_values(['Category', 'Item'])

    # Display each category separately for better readability
    for category in display_df['Category'].unique():
        print(f"\n📌 {category}:")
        cat_df = display_df[display_df['Category'] == category][['Item', 'Quantity', 'Unit', 'Unit_Price', 'Total_Price']]
        print(cat_df.to_string(index=False))
        print("-" * 60)

    # Export option
    print("\n" + "="*80)
    print("💾 To export BOQ to Excel, click the button below")

# Create export button
export_button = widgets.Button(
    description='Export BOQ to Excel',
    button_style='info',
    icon='download',
    layout=widgets.Layout(width='200px', height='40px')
)

def export_boq(b):
    with output:
        try:
            # Create filename with timestamp
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"Solar_BOQ_{timestamp}.xlsx"

            # Create Excel writer
            with pd.ExcelWriter(filename, engine='openpyxl') as writer:
                # Write detailed BOQ
                boq_df.to_excel(writer, sheet_name='Detailed BOQ', index=False)

                # Write summary by category
                category_summary = boq_df.groupby('Category')['Total_Price'].agg(['sum', 'count']).reset_index()
                category_summary.columns = ['Category', 'Total Cost (₱)', 'Number of Items']
                category_summary['Percentage'] = (category_summary['Total Cost (₱)'] / boq_df['Total_Price'].sum() * 100).round(1)
                category_summary.to_excel(writer, sheet_name='Category Summary', index=False)

                # Write system specifications
                specs_df = pd.DataFrame({
                    'Parameter': ['System Type', 'Total Panels', 'Panel Power', 'Total DC Capacity',
                                 'Inverter Power', 'Battery Capacity', 'Daily Consumption', 'Location'],
                    'Value': [system_type.value, num_panels, f"{panel_power}W", f"{total_panel_capacity:.2f}kW",
                             f"{selected_inverter_power}kW",
                             f"{battery_info['total_capacity_kwh']:.2f}kWh" if battery_info else 'N/A',
                             f"{daily_cons:.2f}kWh" if daily_cons else 'N/A', location.value]
                })
                specs_df.to_excel(writer, sheet_name='System Specs', index=False)

            print(f"✅ BOQ exported successfully to: {filename}")

            # In Colab, offer download link
            if IN_COLAB:
                from google.colab import files
                files.download(filename)
                print("📥 Download started.")

        except Exception as e:
            print(f"❌ Error exporting BOQ: {str(e)}")

export_button.on_click(export_boq)

# %% [markdown]
# ## System Sizing Functions (Revised Order)

# %%
def select_inverter(required_ac_power_kw, preferred_brand, preferred_type='All'):
    """
    Select appropriate inverter based on required AC power, brand, and type.
    """
    # Filter by type
    if preferred_type == 'All':
        inv_filtered = inverters
    else:
        inv_filtered = inverters[inverters['Type'] == preferred_type]

    # First try preferred brand
    preferred_inverters = inv_filtered[inv_filtered['Brand'] == preferred_brand]
    suitable_preferred = preferred_inverters[preferred_inverters['Power_kW'] >= required_ac_power_kw]

    if len(suitable_preferred) > 0:
        return suitable_preferred.iloc[0]

    # If preferred brand doesn't have suitable inverter, find any suitable of the selected type
    suitable_any = inv_filtered[inv_filtered['Power_kW'] >= required_ac_power_kw]
    if len(suitable_any) > 0:
        return suitable_any.iloc[0]

    # If still none, return the largest available of the selected type (or any if type=All)
    if len(inv_filtered) > 0:
        return inv_filtered.iloc[-1]
    else:
        # If no inverter of selected type, fallback to any inverter
        return inverters.iloc[-1]

def select_batteries(battery_capacity_kwh, system_voltage, preferred_brand, preferred_type='All'):
    """
    Select appropriate batteries based on capacity, voltage, brand, and type.
    """
    if battery_capacity_kwh == 0:
        return None

    # Filter by type
    if preferred_type == 'All':
        batt_filtered = batteries
    else:
        batt_filtered = batteries[batteries['Type'] == preferred_type]

    # Try preferred brand
    preferred_battery = batt_filtered[batt_filtered['Brand'] == preferred_brand]
    if len(preferred_battery) > 0:
        selected_battery = preferred_battery.iloc[0]
    else:
        # If preferred brand not available in this type, take first of type
        if len(batt_filtered) > 0:
            selected_battery = batt_filtered.iloc[0]
        else:
            # Fallback to any battery
            selected_battery = batteries.iloc[0]

    # Calculate number of batteries needed
    num_batteries = np.ceil(battery_capacity_kwh / selected_battery['Capacity_kWh'])

    # Check voltage compatibility
    if selected_battery['Voltage_V'] != system_voltage:
        if system_voltage % selected_battery['Voltage_V'] == 0:
            # Can be connected in series
            num_series = int(system_voltage / selected_battery['Voltage_V'])
            num_parallel = int(np.ceil(num_batteries / num_series))
            num_batteries = num_series * num_parallel

    return {
        'battery': selected_battery,
        'num_batteries': int(num_batteries),
        'total_capacity_kwh': num_batteries * selected_battery['Capacity_kWh']
    }

def calculate_panels(solar_dc_kw, panel_power_w):
    """Calculate number of panels needed for given DC kW"""
    num_panels = int(np.ceil(solar_dc_kw * 1000 / panel_power_w))
    total_panel_capacity = num_panels * panel_power_w / 1000
    return num_panels, total_panel_capacity

# %% [markdown]
# ## Visualization Functions

# %%
def create_system_diagram(inverter_power_kw, solar_dc_kw, num_panels, battery_info,
                         daily_consumption_val, peak_load_val, system_type_val):
    """
    Create a simple system diagram
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Energy flow diagram
    labels = ['Solar Panels (DC)', 'Inverter (AC)', 'Load']

    # Determine load value
    if peak_load_val > 0:
        load_value = peak_load_val * 1000
    else:
        load_value = daily_consumption_val * 1000 / 5  # rough average

    sizes = [solar_dc_kw*1000, inverter_power_kw*1000, load_value]

    if system_type_val in ['Off-Grid', 'Hybrid'] and battery_info is not None:
        labels.append('Battery')
        sizes.append(battery_info['total_capacity_kwh']*1000)
    elif system_type_val == 'On-Grid':
        labels.append('Grid')
        sizes.append(solar_dc_kw*500)  # Placeholder for grid connection

    colors = ['gold', 'lightblue', 'lightgreen', 'lightcoral', 'lightgray']

    ax1.pie(sizes, labels=labels, colors=colors[:len(sizes)], autopct='%1.1f%%', startangle=90)
    ax1.set_title('System Component Ratings (W or Wh)')

    # Battery configuration
    if battery_info is not None:
        battery_data = pd.DataFrame({
            'Metric': ['Capacity (kWh)', 'Voltage (V)', 'Number of Batteries', 'Cycle Life', 'Depth of Discharge'],
            'Value': [f"{battery_info['total_capacity_kwh']:.1f}",
                     battery_info['battery']['Voltage_V'],
                     battery_info['num_batteries'],
                     battery_info['battery']['Cycle_Life'],
                     f"{battery_info['battery']['DoD']*100:.0f}%"]
        })

        ax2.axis('tight')
        ax2.axis('off')
        table = ax2.table(cellText=battery_data.values, colLabels=battery_data.columns,
                         cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 1.5)
        ax2.set_title('Battery Configuration')
    else:
        ax2.text(0.5, 0.5, 'No Battery Required\n(On-Grid System)',
                horizontalalignment='center', verticalalignment='center',
                transform=ax2.transAxes, fontsize=12)
        ax2.set_title('Battery Configuration')
        ax2.axis('off')

    plt.tight_layout()
    plt.show()

def create_boq_chart(boq_df):
    """
    Create a bar chart showing cost breakdown by category
    """
    category_totals = boq_df.groupby('Category')['Total_Price'].sum()

    fig, ax = plt.subplots(figsize=(12, 6))

    # Create bar chart
    bars = ax.bar(range(len(category_totals)), category_totals.values)

    # Customize colors
    colors = plt.cm.Set3(np.linspace(0, 1, len(category_totals)))
    for bar, color in zip(bars, colors):
        bar.set_color(color)

    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars, category_totals.values)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'₱{val:,.0f}', ha='center', va='bottom', rotation=0, fontsize=9)

    # Customize chart
    ax.set_xlabel('Category')
    ax.set_ylabel('Cost (₱)')
    ax.set_title('Solar PV System - Cost Breakdown by Category')
    ax.set_xticks(range(len(category_totals)))
    ax.set_xticklabels(category_totals.index, rotation=45, ha='right')

    # Add grid for better readability
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

# %% [markdown]
# ## Create User Interface

# %%
# Create input widgets
customer_type = widgets.Dropdown(
    options=customer_types,
    value='Residential',
    description='Customer Type:',
    style={'description_width': 'initial'}
)

system_type = widgets.Dropdown(
    options=system_types,
    value='On-Grid',
    description='System Type:',
    style={'description_width': 'initial'}
)

dc_voltage = widgets.Dropdown(
    options=system_voltages,
    value=48,
    description='DC Voltage (V):',
    style={'description_width': 'initial'}
)

input_method = widgets.RadioButtons(
    options=['Daily Consumption (kWh)', 'Monthly Consumption (kWh)', 'Peak Load (kW)'],
    value='Daily Consumption (kWh)',
    description='Input Method:',
    style={'description_width': 'initial'}
)

daily_consumption = widgets.FloatText(
    value=10,
    description='Daily Consumption (kWh):',
    disabled=False,
    style={'description_width': 'initial'}
)

monthly_consumption = widgets.FloatText(
    value=300,
    description='Monthly Consumption (kWh):',
    disabled=True,
    style={'description_width': 'initial'}
)

peak_load = widgets.FloatText(
    value=2,
    description='Peak Load (kW):',
    disabled=True,
    style={'description_width': 'initial'}
)

autonomy_days = widgets.IntSlider(
    value=1,
    min=1,
    max=7,
    step=1,
    description='Autonomy Days:',
    style={'description_width': 'initial'}
)

ac_dc_ratio = widgets.FloatSlider(
    value=1.2,
    min=1.0,
    max=1.5,
    step=0.05,
    description='AC/DC Ratio:',
    style={'description_width': 'initial'},
    readout_format='.2f'
)

# Roof type selection for mounting structure
roof_type = widgets.Dropdown(
    options=['tile', 'metal', 'concrete'],
    value='tile',
    description='Roof Type:',
    style={'description_width': 'initial'}
)

# Distance to grid connection point
grid_distance = widgets.IntSlider(
    value=15,
    min=5,
    max=50,
    step=5,
    description='Grid Distance (m):',
    style={'description_width': 'initial'}
)

# --- Type dropdowns (fixed to avoid sorting errors) ---
# Convert types to strings, drop NaNs, then sort
panel_type_options = ['All'] + sorted(solar_panels['Type'].dropna().astype(str).unique())
panel_type = widgets.Dropdown(
    options=panel_type_options,
    value='All',
    description='Panel Type:',
    style={'description_width': 'initial'}
)

inverter_type_options = ['All'] + sorted(inverters['Type'].dropna().astype(str).unique())
inverter_type = widgets.Dropdown(
    options=inverter_type_options,
    value='All',
    description='Inverter Type:',
    style={'description_width': 'initial'}
)

battery_type_options = ['All'] + sorted(batteries['Type'].dropna().astype(str).unique())
battery_type = widgets.Dropdown(
    options=battery_type_options,
    value='All',
    description='Battery Type:',
    style={'description_width': 'initial'}
)

# Brand dropdowns (will be dynamically filtered)
panel_brand = widgets.Dropdown(
    options=solar_panels['Brand'].unique(),
    value=solar_panels['Brand'].iloc[0] if len(solar_panels) > 0 else 'JinkoSolar',
    description='Panel Brand:',
    style={'description_width': 'initial'}
)

inverter_brand = widgets.Dropdown(
    options=inverters['Brand'].unique(),
    value=inverters['Brand'].iloc[0] if len(inverters) > 0 else 'Growatt',
    description='Inverter Brand:',
    style={'description_width': 'initial'}
)

battery_brand = widgets.Dropdown(
    options=batteries['Brand'].unique(),
    value=batteries['Brand'].iloc[0] if len(batteries) > 0 else 'Pylontech',
    description='Battery Brand:',
    style={'description_width': 'initial'}
)

# --- Functions to update brand options based on type ---
def update_panel_brand_options(*args):
    if panel_type.value == 'All':
        panel_brand.options = solar_panels['Brand'].unique()
    else:
        brands = solar_panels[solar_panels['Type'] == panel_type.value]['Brand'].unique()
        panel_brand.options = brands
    if panel_brand.value not in panel_brand.options:
        panel_brand.value = panel_brand.options[0] if len(panel_brand.options) > 0 else ''

def update_inverter_brand_options(*args):
    if inverter_type.value == 'All':
        inverter_brand.options = inverters['Brand'].unique()
    else:
        brands = inverters[inverters['Type'] == inverter_type.value]['Brand'].unique()
        inverter_brand.options = brands
    if inverter_brand.value not in inverter_brand.options:
        inverter_brand.value = inverter_brand.options[0] if len(inverter_brand.options) > 0 else ''

def update_battery_brand_options(*args):
    if battery_type.value == 'All':
        battery_brand.options = batteries['Brand'].unique()
    else:
        brands = batteries[batteries['Type'] == battery_type.value]['Brand'].unique()
        battery_brand.options = brands
    if battery_brand.value not in battery_brand.options:
        battery_brand.value = battery_brand.options[0] if len(battery_brand.options) > 0 else ''

# Attach observers
panel_type.observe(update_panel_brand_options, names='value')
inverter_type.observe(update_inverter_brand_options, names='value')
battery_type.observe(update_battery_brand_options, names='value')

# Initialize brand options
update_panel_brand_options()
update_inverter_brand_options()
update_battery_brand_options()

# Location and sun hours
location = widgets.Text(
    value='Manila',
    description='Location:',
    style={'description_width': 'initial'}
)

peak_sun_hours = widgets.FloatSlider(
    value=5,
    min=3,
    max=7,
    step=0.1,
    description='Peak Sun Hours:',
    style={'description_width': 'initial'}
)

# Update function for input method
def update_input_method(change):
    method = change['new']
    daily_consumption.disabled = (method != 'Daily Consumption (kWh)')
    monthly_consumption.disabled = (method != 'Monthly Consumption (kWh)')
    peak_load.disabled = (method != 'Peak Load (kW)')

input_method.observe(update_input_method, names='value')

# Create output area for results
output = widgets.Output()

# Define the calculation function with revised order and BOQ
def calculate_system(b):
    global boq_df, num_panels, panel_power, selected_panel, inverter_info, selected_inverter_power, total_panel_capacity, battery_info, daily_cons

    with output:
        clear_output(wait=True)

        # Get input values
        method = input_method.value
        cons_daily = daily_consumption.value if method == 'Daily Consumption (kWh)' else None
        cons_monthly = monthly_consumption.value if method == 'Monthly Consumption (kWh)' else None
        load = peak_load.value if method == 'Peak Load (kW)' else None

        # Convert monthly to daily if needed
        if cons_monthly is not None and cons_monthly > 0:
            daily_cons = cons_monthly / 30  # approximate
            cons_display = cons_monthly
        else:
            daily_cons = cons_daily
            cons_display = cons_daily

        # Validate inputs
        if (daily_cons is None or daily_cons == 0) and (load is None or load == 0):
            print("⚠️ Please enter either daily/monthly consumption or peak load")
            return

        try:
            # --- ASSUMPTIONS ---
            print("="*70)
            print("🔢 SOLAR PV SYSTEM SIZING - STEP-BY-STEP CALCULATION")
            print("="*70)
            print("\n📌 ASSUMPTIONS:")
            print("   - System efficiency (accounting for losses): 75%")
            print("   - Safety factor for peak load sizing: 25%")
            print("   - Battery Depth of Discharge (DoD): 90%")
            print(f"   - Autonomy days: {autonomy_days.value} day(s)")
            print(f"   - AC/DC ratio (DC array / inverter AC): {ac_dc_ratio.value:.2f}")

            # --- INPUT SUMMARY ---
            print("\n📥 INPUT SUMMARY:")
            print(f"   - Customer Type: {customer_type.value}")
            print(f"   - System Type: {system_type.value}")
            print(f"   - DC System Voltage: {dc_voltage.value} V")
            print(f"   - Location: {location.value} (Peak Sun Hours: {peak_sun_hours.value} h/day)")
            print(f"   - Roof Type: {roof_type.value}")
            print(f"   - Grid Connection Distance: {grid_distance.value} m")
            print(f"   - Input Method: {method}")
            if method == 'Daily Consumption (kWh)':
                print(f"   - Daily Consumption: {daily_cons} kWh")
            elif method == 'Monthly Consumption (kWh)':
                print(f"   - Monthly Consumption: {cons_display} kWh → Daily (approx): {daily_cons:.2f} kWh")
            else:
                print(f"   - Peak Load: {load} kW")

            # --- Get selected panel with type filter ---
            if panel_type.value == 'All':
                panel_candidates = solar_panels[solar_panels['Brand'] == panel_brand.value]
            else:
                panel_candidates = solar_panels[(solar_panels['Brand'] == panel_brand.value) &
                                                (solar_panels['Type'] == panel_type.value)]
            if len(panel_candidates) == 0:
                # Fallback to any panel of the same brand
                panel_candidates = solar_panels[solar_panels['Brand'] == panel_brand.value]
            selected_panel = panel_candidates.iloc[0]
            panel_power = float(selected_panel['Power_Range_W'].split('-')[1])
            print(f"   - Selected Solar Panel: {panel_brand.value} ({selected_panel['Type']}) - {panel_power} W")
            print(f"   - Selected Inverter: {inverter_brand.value} ({inverter_type.value})")
            print(f"   - Selected Battery: {battery_brand.value} ({battery_type.value})")

            # --- STEP 1: DETERMINE INVERTER AC SIZE ---
            print("\n" + "-"*70)
            print("STEP 1: DETERMINE INVERTER AC SIZE")
            print("-"*70)

            if load is not None and load > 0:
                # Sizing based on peak load
                inverter_ac_kw = load * 1.25
                print(f"   Based on peak load with 25% safety factor:")
                print(f"   Inverter AC size = {load} kW × 1.25 = {inverter_ac_kw:.2f} kW")
            else:
                # Sizing based on daily consumption
                daily_energy_needed = daily_cons / 0.75  # accounting for losses
                avg_power_kw = daily_energy_needed / peak_sun_hours.value
                inverter_ac_kw = avg_power_kw * 1.25  # add safety margin for peak loads
                print(f"   Based on daily consumption:")
                print(f"   Daily energy needed (after losses) = {daily_cons:.2f} kWh / 0.75 = {daily_energy_needed:.2f} kWh")
                print(f"   Average power during {peak_sun_hours.value} peak sun hours = {daily_energy_needed:.2f} kWh / {peak_sun_hours.value} h = {avg_power_kw:.2f} kW")
                print(f"   Applying 25% safety margin for peak loads: {avg_power_kw:.2f} kW × 1.25 = {inverter_ac_kw:.2f} kW")

            print(f"\n✅ Required Inverter AC Power: {inverter_ac_kw:.2f} kW")

            # Select inverter with type
            inverter_info = select_inverter(inverter_ac_kw, inverter_brand.value, inverter_type.value)
            selected_inverter_power = inverter_info['Power_kW']
            print(f"\n   Selected inverter: {inverter_info['Brand']} ({inverter_info['Type']}) - {selected_inverter_power} kW")
            print(f"   (Closest available model meeting or exceeding {inverter_ac_kw:.2f} kW)")
            print(f"   Efficiency: {inverter_info['Efficiency']*100:.1f}%")

            # --- STEP 2: BATTERY SIZING (if applicable) ---
            battery_info = None
            battery_capacity_kwh = 0
            if system_type.value in ['Off-Grid', 'Hybrid'] and daily_cons is not None:
                print("\n" + "-"*70)
                print("STEP 2: BATTERY SIZING")
                print("-"*70)
                battery_capacity_kwh = (daily_cons * autonomy_days.value) / (0.9 * 0.75)
                print(f"   Formula: Battery capacity (kWh) = (Daily consumption × Autonomy days) / (DoD × Efficiency)")
                print(f"   = ({daily_cons:.2f} kWh × {autonomy_days.value} day) / (0.9 × 0.75)")
                print(f"   = {daily_cons*autonomy_days.value:.2f} / 0.675 = {battery_capacity_kwh:.2f} kWh")

                battery_info = select_batteries(battery_capacity_kwh, dc_voltage.value, battery_brand.value, battery_type.value)
                if battery_info:
                    batt = battery_info['battery']
                    print(f"\n   Selected battery: {batt['Brand']} ({batt['Type']}) - {batt['Capacity_kWh']} kWh @ {batt['Voltage_V']}V")
                    print(f"   Number of batteries needed = ceil( {battery_capacity_kwh:.2f} kWh / {batt['Capacity_kWh']} kWh )")
                    num_basic = int(np.ceil(battery_capacity_kwh / batt['Capacity_kWh']))
                    print(f"   = ceil( {battery_capacity_kwh/batt['Capacity_kWh']:.2f} ) = {num_basic} batteries")

                    if batt['Voltage_V'] != dc_voltage.value:
                        print(f"\n   ⚡ Voltage adjustment: Battery voltage ({batt['Voltage_V']}V) differs from system voltage ({dc_voltage.value}V).")
                        if dc_voltage.value % batt['Voltage_V'] == 0:
                            series = int(dc_voltage.value / batt['Voltage_V'])
                            parallel = int(np.ceil(num_basic / series))
                            total_batteries = series * parallel
                            print(f"   Configured as {series} in series, {parallel} in parallel → total {total_batteries} batteries")
                            battery_info['num_batteries'] = total_batteries
                            battery_info['total_capacity_kwh'] = total_batteries * batt['Capacity_kWh']

                    print(f"   Total battery capacity: {battery_info['total_capacity_kwh']:.2f} kWh")

            # --- STEP 3: SIZE SOLAR PANELS (DC) ---
            print("\n" + "-"*70)
            print("STEP 3: SIZE SOLAR PANELS (DC ARRAY)")
            print("-"*70)

            # Solar DC size = Inverter AC size × AC/DC ratio
            solar_dc_kw = selected_inverter_power * ac_dc_ratio.value
            print(f"   Formula: DC array size = Inverter AC power × AC/DC ratio")
            print(f"   = {selected_inverter_power:.2f} kW × {ac_dc_ratio.value:.2f} = {solar_dc_kw:.2f} kW")

            num_panels, total_panel_capacity = calculate_panels(solar_dc_kw, panel_power)
            print(f"\n   Number of panels needed = ceil( {solar_dc_kw:.2f} kW × 1000 / {panel_power} W )")
            print(f"   = ceil( {solar_dc_kw*1000/panel_power:.2f} ) = {num_panels} panels")
            print(f"   Total panel capacity = {num_panels} × {panel_power} W = {total_panel_capacity:.2f} kW")

            # Check if solar production meets daily consumption (for reference)
            daily_production = total_panel_capacity * peak_sun_hours.value * 0.75
            print(f"\n   Estimated daily energy production: {total_panel_capacity:.2f} kW × {peak_sun_hours.value} h × 0.75 = {daily_production:.2f} kWh")
            if daily_cons is not None:
                if daily_production >= daily_cons:
                    print(f"   ✅ Production meets or exceeds daily consumption of {daily_cons:.2f} kWh")
                else:
                    print(f"   ⚠️ Production ({daily_production:.2f} kWh) is less than daily consumption ({daily_cons:.2f} kWh). Consider increasing AC/DC ratio or panel size.")

            # --- STEP 4: DETAILED BILL OF QUANTITIES ---
            print("\n" + "-"*70)
            print("STEP 4: DETAILED BILL OF QUANTITIES (BOQ)")
            print("-"*70)

            # Calculate BOQ
            boq_df = calculate_boq(num_panels, panel_power, selected_inverter_power, battery_info, system_type.value, daily_cons, selected_panel, inverter_info, roof_type.value, grid_distance.value)

            # Display BOQ summary
            display_boq_summary(boq_df)

            # Create BOQ chart
            print("\n📊 COST BREAKDOWN CHART:")
            create_boq_chart(boq_df)

            # --- SPACE REQUIREMENTS ---
            print("\n" + "-"*70)
            print("📐 SPACE REQUIREMENTS")
            print("-"*70)
            panel_area = num_panels * 1.7
            print(f"   Roof area needed: {num_panels} panels × 1.7 m² = {panel_area:.1f} m²")
            if battery_info:
                battery_area = max(2, battery_info['num_batteries'] * 0.5)
                print(f"   Battery room area: ~{battery_area:.1f} m²")

            # --- TECHNICAL SPECIFICATIONS ---
            print("\n" + "-"*70)
            print("🔧 TECHNICAL SPECIFICATIONS")
            print("-"*70)
            print(f"   Array Configuration:")
            num_strings = max(1, int(np.ceil(num_panels / 15)))
            panels_per_string = int(np.ceil(num_panels / num_strings))
            print(f"   - {num_strings} strings × {panels_per_string} panels per string")
            print(f"   - String voltage: {selected_panel['Vmp_V'] * panels_per_string:.1f} V")
            print(f"   - String current: {selected_panel['Imp_A']:.1f} A")
            print(f"   - Total DC current: {selected_panel['Imp_A'] * num_strings:.1f} A")

            if battery_info:
                batt = battery_info['battery']
                if batt['Voltage_V'] != dc_voltage.value:
                    series = int(dc_voltage.value / batt['Voltage_V'])
                    parallel = int(battery_info['num_batteries'] / series)
                    print(f"\n   Battery Configuration:")
                    print(f"   - {series} in series × {parallel} in parallel")
                    print(f"   - System voltage: {dc_voltage.value}V")
                    print(f"   - Total capacity: {battery_info['total_capacity_kwh']:.2f} kWh")

            # --- RECOMMENDATIONS ---
            print("\n" + "-"*70)
            print("💡 RECOMMENDATIONS")
            print("-"*70)
            if system_type.value == 'On-Grid':
                print("✓ On-grid system recommended for areas with stable grid")
                print("✓ Consider net metering with Meralco or local electric cooperative")
                print("✓ No battery required, reducing initial cost by 30-40%")
                print("✓ Payback period typically 4-7 years")
                print("✓ Ensure compliance with utility interconnection requirements")
            elif system_type.value == 'Off-Grid':
                print("✓ Ensure sufficient battery capacity for night time and cloudy days")
                print("✓ Consider backup generator for extended cloudy periods")
                print("✓ Regular battery maintenance required every 3-6 months")
                print("✓ Install energy-efficient appliances to reduce consumption")
                print("✓ Consider adding more panels than calculated for cloudy days")
            else:  # Hybrid
                print("✓ Hybrid system provides flexibility and backup power")
                print("✓ Can operate both on-grid and off-grid")
                print("✓ Optimal for areas with unreliable grid")
                print("✓ Battery backup for critical loads during outages")
                print("✓ Program backup priorities for essential loads only")

            # Create system diagram
            create_system_diagram(
                inverter_power_kw=selected_inverter_power,
                solar_dc_kw=total_panel_capacity,
                num_panels=num_panels,
                battery_info=battery_info,
                daily_consumption_val=daily_cons if daily_cons is not None else 0,
                peak_load_val=load if load is not None else 0,
                system_type_val=system_type.value
            )

            # Display export button
            display(widgets.HBox([export_button], layout=widgets.Layout(justify_content='center', margin='20px 0')))

        except Exception as e:
            print(f"❌ An error occurred: {str(e)}")
            print("Please check your inputs and try again.")

# Create the calculate button
calculate_button = widgets.Button(
    description='Calculate System',
    button_style='success',
    tooltip='Click to calculate system size',
    icon='calculator',
    layout=widgets.Layout(width='200px', height='40px')
)

# Attach the callback function
calculate_button.on_click(calculate_system)

# %% [markdown]
# ## Main Interface

# %%
# Create title with styling
title = widgets.HTML("""
<div style='background-color: #f0f8ff; padding: 20px; border-radius: 10px; margin-bottom: 20px;'>
    <h1 style='color: #0066cc; margin: 0;'>☀️ Solar PV System Sizing Tool</h1>
    <h3 style='color: #004999; margin: 10px 0 0 0;'>Philippines & Chinese Equipment</h3>
    <p style='color: #666; margin: 10px 0 0 0;'>For Residential and Commercial Applications</p>
    <p style='color: #0066cc; margin: 5px 0 0 0;'><strong>New: Detailed Bill of Quantities including all accessories</strong></p>
</div>
""")

# Create input sections with better organization
input_sections = widgets.VBox([
    widgets.HTML("<h4 style='margin-bottom: 5px;'>📋 Customer Information</h4>"),
    widgets.HBox([customer_type, location], layout=widgets.Layout(margin='0 0 10px 0')),

    widgets.HTML("<h4 style='margin-bottom: 5px;'>⚙️ System Parameters</h4>"),
    widgets.HBox([system_type, dc_voltage], layout=widgets.Layout(margin='0 0 10px 0')),
    widgets.HBox([peak_sun_hours, autonomy_days], layout=widgets.Layout(margin='0 0 10px 0')),
    widgets.HBox([ac_dc_ratio], layout=widgets.Layout(margin='0 0 10px 0')),
    widgets.HBox([roof_type, grid_distance], layout=widgets.Layout(margin='0 0 10px 0')),

    widgets.HTML("<h4 style='margin-bottom: 5px;'>📊 Load Information</h4>"),
    input_method,
    widgets.VBox([
        daily_consumption,
        monthly_consumption,
        peak_load
    ], layout=widgets.Layout(margin='10px 0')),

    widgets.HTML("<h4 style='margin-bottom: 5px;'>🏭 Equipment Selection</h4>"),
    widgets.HBox([panel_type, panel_brand], layout=widgets.Layout(margin='0 0 5px 0')),
    widgets.HBox([inverter_type, inverter_brand], layout=widgets.Layout(margin='0 0 5px 0')),
    widgets.HBox([battery_type, battery_brand], layout=widgets.Layout(margin='0 0 10px 0')),
])

# Create the main container
main_container = widgets.VBox([
    title,
    widgets.HTML("<hr>"),
    input_sections,
    widgets.HBox([calculate_button], layout=widgets.Layout(justify_content='center', margin='20px 0')),
    widgets.HTML("<hr>"),
    widgets.HTML("<h3>📊 Calculation Results</h3>"),
    output
])

# Display the interface
display(main_container)

# %% [markdown]
# ## Equipment Reference

# %%
# Create tabs for equipment information
tab = widgets.Tab()
tab.children = [
    widgets.Output(),  # Solar Panels
    widgets.Output(),  # Inverters
    widgets.Output(),  # Batteries
    widgets.Output()   # BOQ Template
]

tab.set_title(0, 'Solar Panels')
tab.set_title(1, 'Inverters')
tab.set_title(2, 'Batteries')
tab.set_title(3, 'BOQ Template')

# Populate the tabs
with tab.children[0]:
    panels_display = solar_panels.copy()
    panels_display = panels_display.rename(columns={'Price_PHP_W': 'Price (₱/W)'})
    display(panels_display)

with tab.children[1]:
    inverters_display = inverters.copy()
    inverters_display = inverters_display.rename(columns={'Price_PHP_kW': 'Price (₱/kW)'})
    display(inverters_display)

with tab.children[2]:
    batteries_display = batteries.copy()
    batteries_display = batteries_display.rename(columns={'Price_PHP_kWh': 'Price (₱/kWh)'})
    display(batteries_display)

with tab.children[3]:
    display(HTML("""
    <div style='padding: 20px; background-color: #f8f9fa; border-radius: 5px;'>
        <h4>📋 BOQ Template Structure</h4>
        <p>The Bill of Quantities includes the following categories:</p>
        <ul>
            <li><strong>Solar Panels</strong> - Main PV modules</li>
            <li><strong>Inverter</strong> - String or hybrid inverter</li>
            <li><strong>Batteries</strong> - Energy storage (if applicable)</li>
            <li><strong>Mounting Structure</strong> - Rails, clamps, L-feet, flashings</li>
            <li><strong>DC Electrical</strong> - PV cables, combiners, MC4 connectors, DC isolators, SPDs</li>
            <li><strong>AC Electrical</strong> - AC cables, breakers, isolators, meters</li>
            <li><strong>Protection & Safety</strong> - Circuit breakers, SPDs, grounding, fire extinguisher</li>
            <li><strong>Monitoring System</strong> - Energy meters, data loggers, sensors</li>
            <li><strong>Consumables</strong> - Cable ties, markers, conduit, sealant</li>
            <li><strong>Labor</strong> - Installation and commissioning</li>
            <li><strong>Logistics</strong> - Delivery and equipment rental</li>
            <li><strong>Permits</strong> - Building permits and interconnection fees</li>
        </ul>
        <p>All prices are in Philippine Peso (₱) and include VAT where applicable.</p>
    </div>
    """))

display(widgets.HTML("<h3>📚 Available Equipment Reference</h3>"))
display(tab)

# %% [markdown]
# ## Quick Start Guide for Google Colab
#
# 1. **Run the cell above** to load the interface.
# 2. **Upload the three Excel files** when prompted. (If you have them already in the Colab environment, skip.)
# 3. **Adjust the input parameters** in the widget panel.
# 4. **Click "Calculate System"** to see the results, detailed BOQ, and charts.
# 5. **Click "Export BOQ to Excel"** to generate an Excel file. The file will be automatically downloaded in Colab.
#
# **Note:** If widgets do not appear interactive, run the following command once in a new cell:
# `!jupyter nbextension enable --py widgetsnbextension`
# Then restart the runtime (Runtime → Restart runtime) and run all cells again.

✅ Equipment data loaded successfully from Excel files.


HTML(value='<h3>📚 Available Equipment Reference</h3>')